# 01. Análisis Exploratorio de Datos (EDA)

> **NOTA TÉCNICA - DATA LEAKAGE:**

> El proceso de Análisis Exploratorio de Datos (EDA) descrito a continuación se ejecuta sobre el **dataset completo (población = 20,640 registros)** para diagnosticar la distribución general. 

> La aplicación de cualquier técnica correctiva dictaminada por este análisis (imputación ponderada, normalización, feature engineering) **deberá calcularse estrictamente utilizando los parámetros del conjunto de entrenamiento (`train_set`)** en la etapa de modelado, preservando un aislamiento total frente al `test_set` para garantizar la validez del modelo.


In [7]:
# Importación de dependencias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.figsize'] = (12, 6)

# Carga de datos completos (Raw)
housing = pd.read_csv('../data/raw/housing/housing.csv')
print(f"Dimensiones del conjunto de datos: {housing.shape[0]:,} observaciones, {housing.shape[1]} variables.")


Dimensiones del conjunto de datos: 20,640 observaciones, 10 variables.


---
## 1. Inspección Estructural y Tipología

Inspección inicial mediante muestreo aleatorio para evitar sesgos de orden (e.g., geográficos o cronológicos) introducidos por la recolección secuencial del censo.

In [8]:
print("=== Muestra Estocástica (5 Registros) ===")
display(housing.sample(5, random_state=42))

print("\n=== Perfil de los Datos ===")
housing.info()


=== Muestra Estocástica (5 Registros) ===


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
20046,-122.38,40.67,10.0,2281.0,444.0,1274.0,438.0,2.2120,65600.0,INLAND
3024,-118.37,33.83,35.0,1207.0,207.0,601.0,213.0,4.7308,353400.0,<1H OCEAN
15663,-117.24,32.72,39.0,3089.0,431.0,1175.0,432.0,7.5925,466700.0,NEAR OCEAN
20484,-118.44,34.05,18.0,4780.0,1192.0,1886.0,1036.0,4.4674,500001.0,<1H OCEAN
9814,-118.44,34.18,33.0,2127.0,414.0,1056.0,391.0,4.3750,286100.0,<1H OCEAN



=== Perfil de los Datos ===
<class 'pandas.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   longitude           20640 non-null  float64
 1   latitude            20640 non-null  float64
 2   housing_median_age  20640 non-null  float64
 3   total_rooms         20640 non-null  float64
 4   total_bedrooms      20433 non-null  float64
 5   population          20640 non-null  float64
 6   households          20640 non-null  float64
 7   median_income       20640 non-null  float64
 8   median_house_value  20640 non-null  float64
 9   ocean_proximity     20640 non-null  str    
dtypes: float64(9), str(1)
memory usage: 1.6 MB


### Observación Técnica 1: Definición de la Granularidad
*   **Unidad de análisis:** El nivel de granularidad corresponde al **distrito censal (block group)**, agrupaciones de cientos o miles de habitantes, y no a unidades residenciales individuales.
*   **Implicación en Feature Engineering:** Variables base como `total_rooms` o `population` referencian parámetros macro sumados por distrito. Su uso crudo induce colinealidad vinculada elásticamente al tamaño físico del bloque. Deben generarse variables relacionales (ej. `rooms_per_household`) para representar correctamente a la vivienda prototípica de cada conglomerado.


---
## 2. Auditoría de Valores Ausentes

Al presentarse observaciones nulas en `total_bedrooms`, se procede a la identificación estadística del mecanismo de pérdida (Ausencia Completamente Aleatoria MCAR o en su defecto, MAR/MNAR) evaluando la distorsión frente al marco general.


In [ ]:
# Segregación condicional por estado del dato
nulos_df = housing[housing['total_bedrooms'].isnull()]
validos_df = housing[housing['total_bedrooms'].notnull()]

print(f"Total de registros con variables ausentes ('total_bedrooms'): {len(nulos_df)} ({len(nulos_df)/len(housing)*100:.2f}% del total).")
print("\n--- EVALUACIÓN DIAGNÓSTICA CONTRA TENDENCIA CENTRAL ---")

metricas_comparativas = ['median_income', 'median_house_value', 'housing_median_age', 'population']

comparacion = pd.DataFrame({
    'Mediana (Dataset Válido)': validos_df[metricas_comparativas].median(),
    'Mediana (Dataset Ausente)': nulos_df[metricas_comparativas].median()
})

display(comparacion.round(2))

print("\nDistribución Relativa Geográfica (ocean_proximity):")
geo_comp = pd.DataFrame({
    'Proporción en Ausentes (%)': nulos_df['ocean_proximity'].value_counts(normalize=True) * 100,
    'Proporción en Válidos (%)': validos_df['ocean_proximity'].value_counts(normalize=True) * 100
}).fillna(0).round(2)
display(geo_comp)


### Observación Técnica 2: Clasificación de Faltantes como MCAR
*   **Diagnóstico:** La comparativa de distribución (estratificación por ingreso, valuación predial y distribución demográfica-geográfica) no arroja varianza estadísticamente significativa entre las observaciones con atributos ausentes frente al estándar.
*   **Clasificación:** El comportamiento corresponde a la figura de **MCAR (Missing Completely At Random)**, indicando fallos administrativos carentes de correlación causal localizable.
*   **Tratamiento:** Se procede a la recomendación de imputación por medidas de Tendencia Central (Mediana) a construirse aisladamente sobre el conjunto de entrenamiento.


---
## 3. Dispersión, Límites de Instrumentos y Valores Atípicos

Evaluación de los límites mecánicos del instrumento de encuesta y cálculo formal de valores atípicos mediante rango intercuartílico (*Tukey IQR*) y desviación absoluta centralizada (*Robust Z-Score/MAD*).

In [ ]:
# 1. Cuantificación de censura por la derecha (Capping)
capped_value = (housing['median_house_value'] >= 500001).sum()
capped_age = (housing['housing_median_age'] >= 52).sum()

print("TOPES EN LA VARIABLE DEPENDIENTE E INDEPENDIENTES (CAPPING):")
print(f"Registros en capping (median_house_value ≥500,001): {capped_value} observables ({capped_value/len(housing)*100:.1f}%)")
print(f"Registros en capping (housing_median_age ≥52): {capped_age} observables ({capped_age/len(housing)*100:.1f}%)\n")

# 2. Resumen Métrico de Outliers (IQR vs MAD Z-Score)
def analyze_outliers(df, column):
    # Tukey IQR
    Q1, Q3 = df[column].quantile([0.25, 0.75])
    IQR = Q3 - Q1
    o_iqr = ((df[column] < Q1 - 1.5 * IQR) | (df[column] > Q3 + 1.5 * IQR)).sum()
    
    # Robust Z-Score (MAD)
    mediana = df[column].median()
    mad = np.median(np.abs(df[column] - mediana)) or 1e-6
    robust_z = 0.6745 * (df[column] - mediana) / mad
    o_z = (np.abs(robust_z) > 3).sum()
    
    return o_iqr, o_z

outlier_report = []
for col in ['total_rooms', 'total_bedrooms', 'population', 'households', 'median_income', 'median_house_value']:
    iqr, rb_z = analyze_outliers(housing.dropna(), col)
    outlier_report.append({'Variable': col, 'Outliers IQR (N / %)': f"{iqr} ({iqr/len(housing)*100:.1f}%)", 'Outliers MAD Z-Score (N / %)': f"{rb_z} ({rb_z/len(housing)*100:.1f}%)"})

display(pd.DataFrame(outlier_report))

# 3. Representación Gráfica (Boxplots)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.boxplot(x=housing['median_income'], ax=axes[0], color='lightgreen')
axes[0].set_title('Distribución Asimétrica Positiva (median_income)')
sns.boxplot(x=housing['population'], ax=axes[1], color='salmon')
axes[1].set_title('Outliers Geográficos Extremos (population)')
sns.boxplot(x=housing['median_house_value'], ax=axes[2], color='skyblue')
axes[2].set_title('Truncamiento Visual / Capping (median_house_value)')
plt.tight_layout()
plt.show()


### Observación Técnica 3: Capping Artificial
El truncamiento abrupto a los **500,001 USD** en `median_house_value` y a los **52 años** en `housing_median_age` restringe estructuralmente la capacidad generalizadora del modelo subyacente. Los modelos algorítmicos serán incapaces de extrapolar varianza para cualquier sector superior. Dependiendo de los requerimientos corporativos, el tratamiento sugiere desestimar o retener estas capturas o implementar pipelines secuenciales.

### Observación Técnica 4: Distribución de las Variables Económicas
El reporte de Z-Score confirma asimetrías severas particularmente en los atributos de ingresos e infraestructura (v.g. >2.5% poblacional). Dadas las características biológicas y socioeconómicas reales (distribución desigual real, no error de transcripción), los algoritmos lineales sub-responderán. Son precisas transformaciones de simetrización o preescalamiento (StandardScaler).


---
## 4. Análisis de Asimetría Teórica y Modelos PDF

Validación analítica de la Normalidad Estándar utilizando la Asimetría (Skewness), la Curtosis y la confirmación gráfica de probabilidad Q-Q (Cuantil-Cuantil).

In [ ]:
# Análisis Estadístico de Forma General
forma_st = []
for c in ['housing_median_age', 'total_rooms', 'population', 'median_income', 'median_house_value']:
    s = housing[c].dropna()
    forma_st.append({
        'Feature': c,
        'Desv. Estándar': s.std(),
        'CV Coef. Variación (%)': (s.std() / s.mean()) * 100,
        'Skewness': s.skew(),
        'Kurtosis': s.kurtosis()
    })
display(pd.DataFrame(forma_st).set_index('Feature').round(2))


In [ ]:
# Análisis de Densidades - PDF mediante Kernel Density Estimator (KDE)
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()
for i, c in enumerate(['housing_median_age', 'population', 'median_income', 'median_house_value']):
    sns.histplot(housing[c], kde=True, bins=50, ax=axes[i], line_kws={'linewidth': 2.5}, color='teal')
    axes[i].set_title(f'KDE - {c.upper()}', fontsize=12)
plt.tight_layout()
plt.show()

# Tests Q-Q 
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
stats.probplot(housing['median_income'], dist="norm", plot=axes[0])
axes[0].set_title('Test Probabilidad QQ: median_income')
stats.probplot(housing['median_house_value'], dist="norm", plot=axes[1])
axes[1].set_title('Test Probabilidad QQ: median_house_value')
plt.tight_layout()
plt.show()


### Observación Técnica 5: Limitaciones de Inferencia Paramétrica Pura
Ninguna de las variables primarias cumple el postulado de normalidad (Gaussian distribution). Atributos funcionales como `population` detentan un Skewness rotundo superior a 4.0, acompañado de desviaciones sistemáticas comprobables en las colas derechas de los Tests Q-Q. Obligatoriedad consecuente de Estandarizaciones Lineales post-splitting.


---
## 5. Correlación Bivariada y Comprobación de Multicolinealidad

Se somete el grado de asociación paramétrica de todas las variables numéricas a métricas combinadas (Spearman vs Pearson) para evaluar sensibilidad de rango de magnitud estricta versus monotonicidad de clase.

In [ ]:
num_housing = housing.select_dtypes(include=[np.number])
corr_p = num_housing.corr(method='pearson')
corr_s = num_housing.corr(method='spearman')

fig, axes = plt.subplots(1, 2, figsize=(20, 8))
mask = np.triu(np.ones_like(corr_p, dtype=bool))
sns.heatmap(corr_p, mask=mask, annot=True, fmt='.2f', cmap='viridis', center=0, vmin=-1, vmax=1, ax=axes[0])
axes[0].set_title('MATRIZ DE PEARSON (Correlación Lineal Continua)')
sns.heatmap(corr_s, mask=mask, annot=True, fmt='.2f', cmap='magma', center=0, vmin=-1, vmax=1, ax=axes[1])
axes[1].set_title('MATRIZ DE SPEARMAN (Correlación sobre Rangos Robustos)')
plt.tight_layout()
plt.show()

# Rendimiento lineal frente a la variable objetivo
print("FACTORES PREDICTIVOS SOBRE 'median_house_value':")
predictores = pd.DataFrame({'Coeficiente Lineal (Pearson)': corr_p['median_house_value'], 'Coeficiente Rango Monotónico (Spearman)': corr_s['median_house_value']})
display(predictores.sort_values(by='Coeficiente Rango Monotónico (Spearman)', ascending=False).round(3))


### Observación Técnica 6: Multicolinealidad Positiva Extrema en Dimensiones Espaciales
El análisis diagnostica dependencia directa y colinealidad (r > 0.85) entre las agrupaciones numéricas dependientes del área (`total_rooms`, `total_bedrooms`, `population` y `households`). Estas variables en conjunto miden invariables escalares sobre el área geográfica.
*   **Procedimiento Técnico Requerido:** Es inviable mantener dichas variables independientes puras en métodos regresivos sin penalizaciones matemáticas. Su colinealidad impone aplicar un proceso de Feature Engineering que descomponga el censo local, generando las siguientes nuevas características (por densidad): `rooms_per_household`, `bedrooms_per_room`, y `population_per_household`.


---
## 6. Variables Nominales y Dispersión Geoespacial
Evaluación final del comportamiento dependiente a partir de las coordenadas bidimensionales referenciales.

In [ ]:
plt.figure(figsize=(10, 7))
scatter = plt.scatter(x=housing['longitude'], y=housing['latitude'], alpha=0.5, 
            s=housing['population']/80, label='Dimensión (Demografía)',
            c=housing['median_house_value'], cmap=plt.get_cmap('jet'))

plt.colorbar(scatter, label='Escala (Valor Mediano)')
plt.xlabel('Eje X Longitudinal')
plt.ylabel('Eje Y Latitudinal')
plt.title('Representación Mínima: Densidad y Precio vs Territorio Californiano', fontsize=12)
plt.legend()
plt.show()


### Observación Técnica 7: Relevancia Categórica Litoral
El mapa certifica el apalancamiento directo frente al atractor natural "Océano" (Variables nominales `ISLAND`, `NEAR OCEAN`, `NEAR BAY`), reduciendo drásticamente el impacto conforme los coeficientes latitud-longitud ingresan al entorno desértico ("INLAND").

---
## 7. Procedimientos Post-EDA

Consolidado del diagnóstico exploratorio ejecutado sobre el 100% poblacional. Las siguientes operaciones proceden en la fase de `Data Preprocessing Pipeline` a someterse explícitamente y exclusivamente en los parámetros del archivo `train_set.csv`.

1.  **Tratamiento de Nulos:** La condición *MCAR* avala el uso generalizado de un SimpleImputer con política tendencial local (Mediana).
2.  **Multicolinealidad:** Esfuerzo mandatorio de Feature Engineering para proveer combinadores sintéticos (`bedrooms_per_room`, `rooms_per_household`).
3.  **Encapsulación de Señales Normales:** Reducción analítica con Standard Scaler para reducir asimetrías de rangos en vectores biológicos dispersos.
4.  **Binarización:** Manejo general `OneHotEncoder` de los dictámenes costeros en la variable binaria unida con control residual `handle_unknown="ignore"`. 
